# Lesson 5: Python Magic Methods

## Learning Objectives

By the end of this lesson, you will be able to:
- Explain what magic methods (dunder methods) are and why they are used.
- Implement `__str__` and `__repr__` to provide useful string representations of your objects.
- Implement `__len__` to allow the built-in `len()` function to work on your custom objects.
- Implement `__getitem__` to enable accessing parts of your object using square bracket notation (e.g., `obj[key]`)
- Recognize how these methods make custom objects behave more like Python's built-in types.

## Why This Topic Matters

Have you ever wondered why you can call `len()` on a list, or why `print()` on a dictionary gives a nicely formatted output? This behavior isn't hardcoded into Python for just a few types. It's enabled by a system of special methods that you can add to your own classes.

These methods are called **magic methods** or **dunder methods** (for the **d**ouble **under**scores that surround their names, like `__init__`).

By implementing dunder methods, you can create objects that feel intuitive and 'Pythonic'. You can define what happens when you add two of your objects together (`+`), what `print(obj)` shows, or what `len(obj)` returns. For data science, this allows us to create classes that represent concepts like datasets, models, or results, and have them interact seamlessly with the rest of the Python ecosystem.

## Intuition-First Explanation: Customizing Your Phone's Buttons

Imagine your smartphone. It has physical buttons like 'Volume Up', 'Volume Down', and 'Power'. When you press 'Volume Up', the phone's operating system doesn't decide what to do; it simply sends a 'volume up was pressed' signal to the currently active app.

- If you're listening to music, the Music app receives the signal and increases the volume.
- If you're in a camera app, it might receive the signal and take a picture.
- If you're on the home screen, the OS itself handles it and shows the volume slider.

Magic methods are like the code inside your app that listens for these system-wide button presses. Python's built-in functions and operators (like `len()`, `print()`, `+`, `[]`) are the 'buttons'. When you use them on one of your objects, Python checks if your object has implemented the corresponding magic method to handle the 'signal'.

- `print(my_obj)` sends a 'get string representation' signal, which is handled by `my_obj.__str__()`.
- `len(my_obj)` sends a 'get length' signal, which is handled by `my_obj.__len__()`.
- `my_obj[5]` sends a 'get item at index 5' signal, which is handled by `my_obj.__getitem__(5)`.

## Core Dunder Methods for Data Science Classes

Let's build a `DataSet` class to represent a collection of data, and we'll add magic methods to it step by step to make it more powerful and intuitive.

In [ ]:
import pandas as pd

class DataSet:
    def __init__(self, name, data):
        self.name = name
        self._data = pd.DataFrame(data)

# --- Without magic methods ---
raw_data = {'colA': [1, 2, 3], 'colB': ['X', 'Y', 'Z']}
my_dataset = DataSet('Initial Data', raw_data)

# What happens when we print it?
print(my_dataset)

That output is not helpful at all! It just tells us the object's type and where it is in memory. Let's fix this.

### 1. String Representation: `__str__` and `__repr__`

- `__str__(self)`: Called by `print(obj)` and `str(obj)`. Should return a user-friendly, readable string describing the object.
- `__repr__(self)`: Called when you just type the object's name in a console or notebook cell. Should return an unambiguous, official string representation of the object, ideally one that could be used to recreate the object (`eval(repr(obj)) == obj`).

**Best practice:** Always implement `__repr__`. If you don't implement `__str__`, Python will fall back and use `__repr__` for printing. So, a simple approach is to make `__repr__` detailed and then, if you want, make `__str__` more user-friendly.

In [ ]:
class DataSet:
    def __init__(self, name, data):
        self.name = name
        self._data = pd.DataFrame(data)
        
    def __repr__(self):
        """The 'official' developer-facing representation."""
        return f"DataSet(name='{self.name}', data=<{len(self._data)} rows, {len(self._data.columns)} cols>)"

    def __str__(self):
        """The 'user-friendly' representation for printing."""
        return f"'{self.name}' DataSet\n---\n{self._data.head(3)}"

# --- With magic methods ---
my_dataset = DataSet('Initial Data', raw_data)

# Now what happens when we print it? (Calls __str__)
print("--- Output of print(my_dataset) ---")
print(my_dataset)

# What happens when we just view the object? (Calls __repr__)
print("\n--- Output of my_dataset in a cell ---")
my_dataset

Much better! Now our object gives us useful information about itself.

### 2. Length of the Object: `__len__`

What is the 'length' of our `DataSet`? It's probably the number of rows. By implementing `__len__`, we can make the built-in `len()` function work on our object.

In [ ]:
class DataSet:
    def __init__(self, name, data):
        self.name = name
        self._data = pd.DataFrame(data)
        
    def __repr__(self):
        return f"DataSet(name='{self.name}', data=<{len(self)} rows, {len(self._data.columns)} cols>)"

    def __str__(self):
        return f"'{self.name}' DataSet ({len(self)} rows)\n---\n{self._data.head(3)}"
    
    def __len__(self):
        """Returns the number of rows in the dataset."""
        print("(Calling __len__...)")
        return len(self._data)

# --- Usage ---
my_dataset = DataSet('Initial Data', raw_data)

# Now we can use the built-in len() function!
print(f"The length of the dataset is: {len(my_dataset)}")

# Our __repr__ and __str__ can also use len(self) for consistency
print("\n--- New print output ---")
print(my_dataset)

### 3. Item Access: `__getitem__`

This is one of the most powerful magic methods. `__getitem__(self, key)` is called whenever you use square brackets `[]` on your object. It lets you define what `my_dataset[something]` means.

For our `DataSet`, we could decide that:
- If the key is a string, we return the corresponding column.
- If the key is an integer, we return the corresponding row.
- If the key is a slice, we return a new `DataSet` with the sliced rows.

In [ ]:
class DataSet:
    def __init__(self, name, data):
        self.name = name
        self._data = pd.DataFrame(data)
        
    def __repr__(self):
        return f"DataSet(name='{self.name}', data=<{len(self)} rows, {len(self._data.columns)} cols>)"

    def __str__(self):
        return f"'{self.name}' DataSet ({len(self)} rows)\n---\n{self._data.head(3)}"
    
    def __len__(self):
        return len(self._data)
    
    def __getitem__(self, key):
        """Defines behavior for the [] operator."""
        print(f"(Calling __getitem__ with key: {key} of type {type(key)}...)")
        if isinstance(key, str):
            # If key is a string, return the column
            return self._data[key]
        elif isinstance(key, int):
            # If key is an integer, return the row
            return self._data.iloc[key]
        elif isinstance(key, slice):
            # If key is a slice, return a new, smaller DataSet object
            sliced_data = self._data[key]
            return DataSet(f"Slice of {self.name}", sliced_data)
        else:
            raise TypeError("Invalid key type for DataSet.")

# --- Usage ---
raw_data_large = {
    'id': range(100, 110),
    'category': ['A', 'B', 'A', 'C', 'B', 'A', 'A', 'C', 'B', 'A'],
    'value': [v * 1.1 for v in range(10)]
}
main_dataset = DataSet('Sales Data', raw_data_large)

print("--- Accessing a column by string ---")
category_col = main_dataset['category']
print(category_col.head())

print("\n--- Accessing a row by integer ---")
third_row = main_dataset[2]
print(third_row)

print("\n--- Accessing a slice ---")
# This calls __getitem__ with a slice object and returns a NEW DataSet
subset_dataset = main_dataset[3:6]
print(subset_dataset) # This print() call will use the __str__ of the new object
print(f"Length of subset: {len(subset_dataset)}")

By implementing `__getitem__`, we've made our `DataSet` object behave very much like a pandas DataFrame, providing an intuitive and powerful interface for our users.

## Other Useful Dunder Methods

There are dozens of dunder methods that let you customize every aspect of your class. Some other common ones include:

- **Comparison**: `__eq__(self, other)` (for `==`), `__ne__` (`!=`), `__lt__` (`<`), `__gt__` (`>`), etc.
- **Arithmetic**: `__add__(self, other)` (for `+`), `__sub__` (`-`), `__mul__` (`*`), etc. You could define `DataSet_A + DataSet_B` to mean concatenating the two datasets.
- **Iteration**: `__iter__(self)` and `__next__(self)` allow your object to be used in a `for` loop.
- **Attribute Access**: `__getattr__(self, name)` and `__setattr__(self, name, value)` let you intercept access to attributes (e.g., `obj.x = 10`).

You don't need to memorize all of them. You just need to know that if you want your object to behave like a built-in type in some way, there's probably a dunder method for it.

## Recap

- **Magic (Dunder) Methods**: Special methods with double underscores (e.g., `__len__`) that allow you to hook into Python's built-in behavior.
- `__str__` and `__repr__`: Provide user-friendly and developer-friendly string representations of your object. A must-have for any custom class.
- `__len__`: Lets the global `len()` function work on your object.
- `__getitem__`: Lets you use square bracket notation (`[]`) for slicing or accessing elements, making your object feel like a container.
- Implementing these methods makes your custom classes more intuitive, readable, and 'Pythonic'.

## Suggested Next Lesson

We've now covered all the core principles and techniques of OOP in Python. The next step is to see how these concepts come together to structure a real-world data science project. In the next lesson, we'll move from individual classes to designing a small, multi-class system to solve a practical problem, focusing on how classes interact with each other.